In [1]:
# ============================================================
# CELL 1: Install FastAPI and dependencies
# uvicorn  = the server that runs FastAPI
# pyngrok  = creates a public URL from Colab (tunnel)
# python-multipart = needed for file upload (images)
# ============================================================

!pip install fastapi uvicorn pyngrok python-multipart \
             transformers torch torchvision \
             google-generativeai pillow requests -q

from google.colab import drive
drive.mount('/content/drive')

print("✅ All packages installed!")

Mounted at /content/drive
✅ All packages installed!


In [2]:
# ============================================================
# CELL 2: Load all three models into memory
# We load once at startup — not on every request
# This is standard production practice (model caching)
# ============================================================

import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)
import google.generativeai as genai
from google.colab import userdata
from PIL import Image
import requests
from io import BytesIO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

# ---- 1. Text model ----
TEXT_MODEL_PATH = "/content/drive/MyDrive/MisinformationGuard/model_v2"
print("⏳ Loading text model...")
tokenizer  = AutoTokenizer.from_pretrained(TEXT_MODEL_PATH)
text_model = AutoModelForSequenceClassification.from_pretrained(TEXT_MODEL_PATH)
text_model = text_model.to(device).eval()
print("✅ Text model loaded")

# ---- 2. Deepfake detector ----
print("⏳ Loading deepfake detector...")
deepfake_pipe = pipeline(
    "image-classification",
    model="dima806/deepfake_vs_real_image_detection",
    device=0 if torch.cuda.is_available() else -1
)
print("✅ Deepfake detector loaded")

# ---- 3. Gemini ----
print("⏳ Connecting to Gemini...")
GEMINI_KEY = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_KEY)
gemini = genai.GenerativeModel("gemini-2.5-flash")
test   = gemini.generate_content("Reply: READY")
print(f"✅ Gemini connected: {test.text.strip()}")

print("\n🚀 All models ready!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Device: cuda

⏳ Loading text model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Text model loaded
⏳ Loading deepfake detector...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


✅ Deepfake detector loaded
⏳ Connecting to Gemini...
✅ Gemini connected: Great! I'm ready too.

What can I help you with?

🚀 All models ready!


In [3]:
# ============================================================
# CELL 3: Core inference functions
# These are called by the API endpoints
# ============================================================

import numpy as np
from datetime import datetime

# ---- Text prediction ----
def predict_text(statement: str) -> dict:
    inputs = tokenizer(
        statement, return_tensors="pt",
        truncation=True, max_length=128, padding=True
    ).to(device)
    with torch.no_grad():
        probs = F.softmax(text_model(**inputs).logits, dim=1)[0].cpu().numpy()
    pred  = int(probs.argmax())
    return {
        "label"     : "FAKE" if pred == 0 else "REAL",
        "confidence": round(float(probs[pred]) * 100, 2),
        "prob_fake" : round(float(probs[0]) * 100, 2),
        "prob_real" : round(float(probs[1]) * 100, 2),
    }

# ---- Image prediction ----
def predict_image_from_pil(img: Image.Image) -> dict:
    results = deepfake_pipe(img.convert("RGB"))
    scores  = {r['label'].upper(): round(r['score']*100, 2) for r in results}
    top     = max(results, key=lambda x: x['score'])
    label   = "FAKE" if "FAKE" in top['label'].upper() else "REAL"
    return {
        "label"     : label,
        "confidence": round(top['score']*100, 2),
        "prob_fake" : scores.get("FAKE", 0),
        "prob_real" : scores.get("REAL", 0),
    }

# ---- Gemini reasoning ----
def gemini_reason(statement: str, text_result: dict,
                  image_result: dict = None) -> str:
    img_section = ""
    if image_result:
        img_section = f"""
IMAGE ANALYSIS (Deepfake Detector):
  Verdict     : {image_result['label']}
  Confidence  : {image_result['confidence']}%
  REAL prob   : {image_result['prob_real']}%
  FAKE prob   : {image_result['prob_fake']}%"""

    prompt = f"""You are an expert fact-checker for Misinformation Guard.

CLAIM: "{statement}"

TEXT MODEL (XLM-RoBERTa):
  Verdict     : {text_result['label']}
  Confidence  : {text_result['confidence']}%
  FAKE prob   : {text_result['prob_fake']}%
  REAL prob   : {text_result['prob_real']}%
{img_section}

Respond in EXACTLY this format:
VERDICT: [LIKELY FAKE / LIKELY REAL / UNCERTAIN]
CONFIDENCE: [HIGH / MEDIUM / LOW]
REASONING: [2-3 sentences]
RED FLAGS: [2-3 bullet points or "None detected"]
HOW TO VERIFY: [1 sentence]
SOURCES: [2-3 source names]"""

    return gemini.generate_content(prompt).text.strip()

# ---- Fusion strategy ----
def fusion_decision(statement: str,
                    image_pil: Image.Image = None) -> dict:
    text_result  = predict_text(statement)
    image_result = predict_image_from_pil(image_pil) if image_pil else None
    ml_conf      = text_result['confidence']

    # Determine which fusion rule applies
    if ml_conf >= 70:
        rule = "Rule 1 — high ML confidence, Gemini for explanation only"
    elif ml_conf >= 55:
        rule = "Rule 2 — medium ML confidence, Gemini verdict + explanation"
    else:
        rule = "Rule 3 — low ML confidence, independent Gemini verdict"

    reasoning = gemini_reason(statement, text_result, image_result)

    # Extract verdict from Gemini response
    verdict_line = [l for l in reasoning.split('\n') if 'VERDICT:' in l]
    final_verdict = "UNCERTAIN"
    if verdict_line:
        v = verdict_line[0].upper()
        if "FAKE" in v:
            final_verdict = "LIKELY FAKE"
        elif "REAL" in v:
            final_verdict = "LIKELY REAL"

    return {
        "statement"     : statement,
        "final_verdict" : final_verdict,
        "fusion_rule"   : rule,
        "text_model"    : text_result,
        "image_model"   : image_result,
        "gemini_analysis": reasoning,
        "timestamp"     : datetime.now().isoformat(),
        "system"        : "Misinformation Guard v2.0"
    }

print("✅ Inference functions ready!")

# Quick test
test_result = predict_text("Vaccines cause autism.")
print(f"\n   Quick test: {test_result}")

✅ Inference functions ready!

   Quick test: {'label': 'FAKE', 'confidence': 53.05, 'prob_fake': 53.05, 'prob_real': 46.95}


In [4]:
# ============================================================
# CELL 4: Define all API endpoints
# ============================================================

from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import uvicorn
import threading
import io

# ---- App setup ----
app = FastAPI(
    title       = "Misinformation Guard API",
    description = """
## 🛡️ Misinformation Guard

A multilingual multimodal misinformation detection system.

### Models
- **Text**: XLM-RoBERTa v2 fine-tuned on LIAR dataset (100+ languages)
- **Image**: Vision Transformer deepfake detector
- **Reasoning**: Gemini 2.5 Flash

### Endpoints
- `POST /analyze/text` — text-only analysis
- `POST /analyze/multimodal` — text + image analysis
- `GET /health` — system health check
    """,
    version     = "2.0.0",
)

# Allow all origins (for testing)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ---- Request/Response models ----
class TextRequest(BaseModel):
    statement : str
    language  : Optional[str] = "auto"

    class Config:
        json_schema_extra = {
            "example": {
                "statement": "Vaccines cause autism in children.",
                "language" : "English"
            }
        }

class AnalysisResponse(BaseModel):
    statement      : str
    final_verdict  : str
    fusion_rule    : str
    text_model     : dict
    image_model    : Optional[dict]
    gemini_analysis: str
    timestamp      : str
    system         : str


# ══════════════════════════════════════════
# ENDPOINT 1: Health check
# ══════════════════════════════════════════
@app.get("/health", tags=["System"])
async def health_check():
    """Check that all models are loaded and ready."""
    return {
        "status"  : "healthy",
        "models"  : {
            "text_model"      : "XLM-RoBERTa v2 — loaded",
            "deepfake_detector": "ViT — loaded",
            "gemini"          : "Gemini 2.5 Flash — connected"
        },
        "device"  : str(device),
        "version" : "2.0.0",
        "timestamp": datetime.now().isoformat()
    }


# ══════════════════════════════════════════
# ENDPOINT 2: Text-only analysis
# ══════════════════════════════════════════
@app.post("/analyze/text",
          response_model=AnalysisResponse,
          tags=["Analysis"])
async def analyze_text(request: TextRequest):
    """
    Analyze a text claim for misinformation.

    Accepts any language supported by XLM-RoBERTa (100+).
    Returns ML verdict, Gemini reasoning, red flags, and sources.
    """
    if not request.statement.strip():
        raise HTTPException(status_code=400,
                            detail="Statement cannot be empty.")
    if len(request.statement) > 2000:
        raise HTTPException(status_code=400,
                            detail="Statement too long (max 2000 chars).")
    try:
        result = fusion_decision(request.statement)
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ══════════════════════════════════════════
# ENDPOINT 3: Multimodal analysis (text + image)
# ══════════════════════════════════════════
@app.post("/analyze/multimodal",
          response_model=AnalysisResponse,
          tags=["Analysis"])
async def analyze_multimodal(
    statement: str = Form(...),
    image    : UploadFile = File(...)
):
    """
    Analyze a text claim AND an accompanying image.

    Runs both the text classifier and deepfake detector,
    then uses Gemini to reason over combined evidence.

    - **statement**: The text claim to analyze
    - **image**: Image file (JPG, PNG, WebP)
    """
    if not statement.strip():
        raise HTTPException(status_code=400,
                            detail="Statement cannot be empty.")

    # Validate image format
    allowed = {"image/jpeg", "image/png", "image/webp", "image/jpg"}
    if image.content_type not in allowed:
        raise HTTPException(status_code=400,
                            detail=f"Image must be JPG/PNG/WebP. Got: {image.content_type}")

    try:
        img_bytes = await image.read()
        img_pil   = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        result    = fusion_decision(statement, image_pil=img_pil)
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ══════════════════════════════════════════
# ENDPOINT 4: Batch text analysis
# ══════════════════════════════════════════
class BatchRequest(BaseModel):
    statements: list[str]

    class Config:
        json_schema_extra = {
            "example": {
                "statements": [
                    "Vaccines cause autism.",
                    "The Eiffel Tower is in Paris."
                ]
            }
        }

@app.post("/analyze/batch",
          tags=["Analysis"])
async def analyze_batch(request: BatchRequest):
    """
    Analyze multiple text claims at once (max 10).
    Uses text model only — no Gemini (to preserve quota).
    Returns quick verdicts for bulk screening.
    """
    if len(request.statements) > 10:
        raise HTTPException(status_code=400,
                            detail="Maximum 10 statements per batch.")
    if not request.statements:
        raise HTTPException(status_code=400,
                            detail="At least one statement required.")

    results = []
    for statement in request.statements:
        if statement.strip():
            result = predict_text(statement)
            results.append({
                "statement" : statement[:100] + "..." if len(statement) > 100 else statement,
                "verdict"   : result['label'],
                "confidence": result['confidence'],
                "prob_fake" : result['prob_fake'],
                "prob_real" : result['prob_real'],
            })

    return {
        "results"   : results,
        "n_analyzed": len(results),
        "note"      : "Batch uses ML model only. Use /analyze/text for full Gemini reasoning.",
        "timestamp" : datetime.now().isoformat()
    }


print("✅ FastAPI app defined!")
print("\n   Endpoints:")
print("   GET  /health")
print("   GET  /docs          ← interactive API documentation")
print("   POST /analyze/text")
print("   POST /analyze/multimodal")
print("   POST /analyze/batch")

✅ FastAPI app defined!

   Endpoints:
   GET  /health
   GET  /docs          ← interactive API documentation
   POST /analyze/text
   POST /analyze/multimodal
   POST /analyze/batch


/tmp/ipykernel_2096/2374829600.py:43: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class TextRequest(BaseModel):
/tmp/ipykernel_2096/2374829600.py:152: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class BatchRequest(BaseModel):


In [5]:
# ============================================================
# CELL 5: Start FastAPI server with public URL via ngrok
# ngrok creates a tunnel so anyone can reach your Colab server
# ============================================================

from pyngrok import ngrok
import uvicorn
import threading
import time

# ---- Get your ngrok token ----
# Free token at: https://dashboard.ngrok.com/get-started/your-authtoken
# Add it to Colab Secrets as NGROK_TOKEN

try:
    NGROK_TOKEN = userdata.get("NGROK_TOKEN")
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ ngrok token loaded from Colab Secrets")
except:
    print("⚠️  No NGROK_TOKEN in secrets — using unauthenticated tunnel")
    print("   Get a free token at: https://dashboard.ngrok.com")
    print("   Add it to Colab Secrets as NGROK_TOKEN")

# ---- Start server in background thread ----
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(3)   # wait for server to start

# ---- Create public tunnel ----
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"🚀 MISINFORMATION GUARD API IS LIVE!")
print(f"{'='*60}")
print(f"\n   Public URL : {public_url}")
print(f"\n   Endpoints  :")
print(f"   {public_url}/docs              ← Open this in your browser!")
print(f"   {public_url}/health")
print(f"   {public_url}/analyze/text")
print(f"   {public_url}/analyze/multimodal")
print(f"   {public_url}/analyze/batch")
print(f"\n   ⚠️  URL is active while this Colab session is open.")
print(f"   ⚠️  Keep this tab open — closing it stops the server.")
print(f"{'='*60}")

✅ ngrok token loaded from Colab Secrets

🚀 MISINFORMATION GUARD API IS LIVE!

   Public URL : NgrokTunnel: "https://solvable-elva-quotably.ngrok-free.dev" -> "http://localhost:8000"

   Endpoints  :
   NgrokTunnel: "https://solvable-elva-quotably.ngrok-free.dev" -> "http://localhost:8000"/docs              ← Open this in your browser!
   NgrokTunnel: "https://solvable-elva-quotably.ngrok-free.dev" -> "http://localhost:8000"/health
   NgrokTunnel: "https://solvable-elva-quotably.ngrok-free.dev" -> "http://localhost:8000"/analyze/text
   NgrokTunnel: "https://solvable-elva-quotably.ngrok-free.dev" -> "http://localhost:8000"/analyze/multimodal
   NgrokTunnel: "https://solvable-elva-quotably.ngrok-free.dev" -> "http://localhost:8000"/analyze/batch

   ⚠️  URL is active while this Colab session is open.
   ⚠️  Keep this tab open — closing it stops the server.


In [6]:
# ============================================================
# FIX: Extract clean URL from NgrokTunnel object
# ============================================================

from pyngrok import ngrok as pyngrok_ngrok

# Get all active tunnels
tunnels = pyngrok_ngrok.get_tunnels()
print("Active tunnels:")
for t in tunnels:
    print(f"  {t.public_url}")

# Extract the https one
BASE_URL = None
for t in tunnels:
    if t.public_url.startswith("https"):
        BASE_URL = t.public_url
        break

if BASE_URL is None and tunnels:
    BASE_URL = tunnels[0].public_url

print(f"\n✅ Clean BASE_URL: {BASE_URL}")
print(f"\n   Open this in browser:")
print(f"   {BASE_URL}/docs")
print(f"\n   If /docs shows 'Not Found', open:")
print(f"   {BASE_URL}/health")
'''

---

### Also — Try These URLs Directly in Your Browser

Based on your ngrok tunnel, try these one by one:
```
https://solvable-elva-quotably.ngrok-free.app/docs
https://solvable-elva-quotably.ngrok-free.app/health'''

Active tunnels:
  https://solvable-elva-quotably.ngrok-free.dev

✅ Clean BASE_URL: https://solvable-elva-quotably.ngrok-free.dev

   Open this in browser:
   https://solvable-elva-quotably.ngrok-free.dev/docs

   If /docs shows 'Not Found', open:
   https://solvable-elva-quotably.ngrok-free.dev/health


'\n\n---\n\n### Also — Try These URLs Directly in Your Browser\n\nBased on your ngrok tunnel, try these one by one:\n```\nhttps://solvable-elva-quotably.ngrok-free.app/docs\nhttps://solvable-elva-quotably.ngrok-free.app/health'

In [7]:
# ============================================================
# CELL 6 (Final): Test all endpoints with clean BASE_URL
# Make sure BASE_URL is set from the fix cell above
# ============================================================

import requests, json

print("=" * 60)
print("🧪 TESTING ALL API ENDPOINTS")
print("=" * 60)

# ---- Test 1: Health ----
print("\n1️⃣  GET /health")
r    = requests.get(f"{BASE_URL}/health")
data = r.json()
print(f"   Status : {data['status']}")
print(f"   Device : {data['device']}")

# ---- Test 2: Text analysis ----
print("\n2️⃣  POST /analyze/text")
r = requests.post(
    f"{BASE_URL}/analyze/text",
    json={"statement": "5G towers spread COVID-19.",
          "language" : "English"},
    headers={"ngrok-skip-browser-warning": "true"}
)
data = r.json()
print(f"   ML     : {data['text_model']['label']} ({data['text_model']['confidence']}%)")
print(f"   FINAL  : {data['final_verdict']}")
print(f"   Rule   : {data['fusion_rule']}")

# ---- Test 3: Batch ----
print("\n3️⃣  POST /analyze/batch")
r = requests.post(
    f"{BASE_URL}/analyze/batch",
    json={"statements": [
        "NASA faked the moon landing.",
        "The UN was founded in 1945.",
        "Bleach cures COVID-19.",
        "Water is H2O.",
    ]},
    headers={"ngrok-skip-browser-warning": "true"}
)
data = r.json()
for item in data['results']:
    icon = "🚨" if item['verdict'] == "FAKE" else "✅"
    print(f"   {icon} {item['verdict']:4s} {item['confidence']:5.1f}%  "
          f"\"{item['statement'][:45]}\"")

print(f"\n{'='*60}")
print(f"✅ API fully working!")
print(f"   Share with supervisor: {BASE_URL}/docs")
print(f"{'='*60}")

🧪 TESTING ALL API ENDPOINTS

1️⃣  GET /health
   Status : healthy
   Device : cuda

2️⃣  POST /analyze/text
   ML     : FAKE (55.36%)
   FINAL  : LIKELY FAKE
   Rule   : Rule 2 — medium ML confidence, Gemini verdict + explanation

3️⃣  POST /analyze/batch
   🚨 FAKE  54.3%  "NASA faked the moon landing."
   🚨 FAKE  51.5%  "The UN was founded in 1945."
   🚨 FAKE  56.8%  "Bleach cures COVID-19."
   🚨 FAKE  55.5%  "Water is H2O."

✅ API fully working!
   Share with supervisor: https://solvable-elva-quotably.ngrok-free.dev/docs


In [8]:
# ============================================================
# STEP 2A: Write app.py — the main server file for HF Spaces
# ============================================================

import os

os.makedirs("/content/hf_deployment", exist_ok=True)

app_code = '''
import os
import io
import torch
import torch.nn.functional as F
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)
import google.generativeai as genai
from PIL import Image
from datetime import datetime
from contextlib import asynccontextmanager

# ── Model globals ──
tokenizer       = None
text_model      = None
deepfake_pipe   = None
gemini_model    = None
device          = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    """Load all models at startup."""
    global tokenizer, text_model, deepfake_pipe, gemini_model, device

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Text model — loaded from HF Hub directly
    print("Loading text model...")
    MODEL_ID   = os.environ.get("TEXT_MODEL_ID", "your-hf-username/misinformation-guard-text")
    tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID)
    text_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
    text_model = text_model.to(device).eval()
    print("Text model ready")

    # Deepfake detector
    print("Loading deepfake detector...")
    deepfake_pipe = pipeline(
        "image-classification",
        model="dima806/deepfake_vs_real_image_detection",
        device=0 if torch.cuda.is_available() else -1
    )
    print("Deepfake detector ready")

    # Gemini
    print("Connecting to Gemini...")
    GEMINI_KEY = os.environ.get("GEMINI_API_KEY")
    if GEMINI_KEY:
        genai.configure(api_key=GEMINI_KEY)
        gemini_model = genai.GenerativeModel("gemini-2.5-flash")
        print("Gemini ready")
    else:
        print("WARNING: No GEMINI_API_KEY set")

    yield   # app runs here

    print("Shutting down...")


app = FastAPI(
    title       = "Misinformation Guard API",
    description = "Multilingual multimodal misinformation detection.",
    version     = "2.0.0",
    lifespan    = lifespan
)

app.add_middleware(
    CORSMiddleware,
    allow_origins  = ["*"],
    allow_methods  = ["*"],
    allow_headers  = ["*"],
)


# ── Inference helpers ──
def _predict_text(statement: str) -> dict:
    inputs = tokenizer(
        statement, return_tensors="pt",
        truncation=True, max_length=128, padding=True
    ).to(device)
    with torch.no_grad():
        probs = F.softmax(text_model(**inputs).logits, dim=1)[0].cpu().numpy()
    pred = int(probs.argmax())
    return {
        "label"     : "FAKE" if pred == 0 else "REAL",
        "confidence": round(float(probs[pred]) * 100, 2),
        "prob_fake" : round(float(probs[0]) * 100, 2),
        "prob_real" : round(float(probs[1]) * 100, 2),
    }

def _predict_image(img: Image.Image) -> dict:
    results = deepfake_pipe(img.convert("RGB"))
    scores  = {r["label"].upper(): round(r["score"] * 100, 2) for r in results}
    top     = max(results, key=lambda x: x["score"])
    label   = "FAKE" if "FAKE" in top["label"].upper() else "REAL"
    return {
        "label"     : label,
        "confidence": round(top["score"] * 100, 2),
        "prob_fake" : scores.get("FAKE", 0),
        "prob_real" : scores.get("REAL", 0),
    }

def _gemini_reason(statement, text_result, image_result=None) -> str:
    if not gemini_model:
        return "Gemini not configured — set GEMINI_API_KEY in Space secrets."
    img_section = ""
    if image_result:
        img_section = f"""
IMAGE ANALYSIS:
  Verdict    : {image_result["label"]}
  Confidence : {image_result["confidence"]}%"""
    prompt = f"""You are an expert fact-checker for Misinformation Guard.

CLAIM: "{statement}"

TEXT MODEL: {text_result["label"]} ({text_result["confidence"]}% confidence)
{img_section}

Reply in EXACTLY this format:
VERDICT: [LIKELY FAKE / LIKELY REAL / UNCERTAIN]
CONFIDENCE: [HIGH / MEDIUM / LOW]
REASONING: [2-3 sentences]
RED FLAGS: [2-3 points or "None detected"]
HOW TO VERIFY: [1 sentence]
SOURCES: [2-3 sources]"""
    return gemini_model.generate_content(prompt).text.strip()

def _full_analysis(statement: str, image_pil=None) -> dict:
    text_result  = _predict_text(statement)
    image_result = _predict_image(image_pil) if image_pil else None
    ml_conf      = text_result["confidence"]
    rule = (
        "Rule 1 — ML high confidence (≥70%)"   if ml_conf >= 70 else
        "Rule 2 — Gemini+context (55-70%)"      if ml_conf >= 55 else
        "Rule 3 — Blind Gemini (<55%)"
    )
    reasoning = _gemini_reason(statement, text_result, image_result)
    verdict_line  = [l for l in reasoning.split("\\n") if "VERDICT:" in l]
    final_verdict = "UNCERTAIN"
    if verdict_line:
        v = verdict_line[0].upper()
        final_verdict = "LIKELY FAKE" if "FAKE" in v else "LIKELY REAL" if "REAL" in v else "UNCERTAIN"
    return {
        "statement"      : statement,
        "final_verdict"  : final_verdict,
        "fusion_rule"    : rule,
        "text_model"     : text_result,
        "image_model"    : image_result,
        "gemini_analysis": reasoning,
        "timestamp"      : datetime.now().isoformat(),
        "system"         : "Misinformation Guard v2.0",
    }


# ── Endpoints ──
class TextRequest(BaseModel):
    statement : str
    language  : Optional[str] = "auto"

class BatchRequest(BaseModel):
    statements: list[str]

@app.get("/health")
async def health():
    return {
        "status"   : "healthy",
        "device"   : str(device),
        "gemini"   : "connected" if gemini_model else "not configured",
        "version"  : "2.0.0",
        "timestamp": datetime.now().isoformat(),
    }

@app.post("/analyze/text")
async def analyze_text(req: TextRequest):
    if not req.statement.strip():
        raise HTTPException(400, "Statement cannot be empty.")
    if len(req.statement) > 2000:
        raise HTTPException(400, "Statement too long (max 2000 chars).")
    try:
        return _full_analysis(req.statement)
    except Exception as e:
        raise HTTPException(500, str(e))

@app.post("/analyze/multimodal")
async def analyze_multimodal(
    statement: str        = Form(...),
    image    : UploadFile = File(...)
):
    if not statement.strip():
        raise HTTPException(400, "Statement cannot be empty.")
    allowed = {"image/jpeg", "image/png", "image/webp"}
    if image.content_type not in allowed:
        raise HTTPException(400, f"Image must be JPG/PNG/WebP.")
    try:
        img_bytes = await image.read()
        img_pil   = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        return _full_analysis(statement, image_pil=img_pil)
    except Exception as e:
        raise HTTPException(500, str(e))

@app.post("/analyze/batch")
async def analyze_batch(req: BatchRequest):
    if len(req.statements) > 10:
        raise HTTPException(400, "Maximum 10 statements per batch.")
    results = []
    for s in req.statements:
        if s.strip():
            r = _predict_text(s)
            results.append({
                "statement" : s[:100] + "..." if len(s) > 100 else s,
                "verdict"   : r["label"],
                "confidence": r["confidence"],
                "prob_fake" : r["prob_fake"],
                "prob_real" : r["prob_real"],
            })
    return {
        "results"   : results,
        "n_analyzed": len(results),
        "timestamp" : datetime.now().isoformat(),
    }
'''

with open("/content/hf_deployment/app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written!")

✅ app.py written!


In [9]:
# ============================================================
# STEP 2B: Write Dockerfile
# HuggingFace Spaces with Docker SDK uses this to build
# ============================================================

dockerfile = """FROM python:3.11-slim

# Install system dependencies
RUN apt-get update && apt-get install -y \\
    git curl gcc g++ \\
    && rm -rf /var/lib/apt/lists/*

# Set working directory
WORKDIR /app

# Copy requirements first (Docker layer caching)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy app
COPY app.py .

# Expose port 7860 (HuggingFace default)
EXPOSE 7860

# Start server
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"]
"""

with open("/content/hf_deployment/Dockerfile", "w") as f:
    f.write(dockerfile)

print("✅ Dockerfile written!")

✅ Dockerfile written!


In [10]:
# ============================================================
# STEP 2C: Python dependencies for HF Spaces
# ============================================================

requirements = """fastapi==0.115.0
uvicorn==0.30.0
python-multipart==0.0.9
transformers==4.44.0
torch==2.4.0
torchvision==0.19.0
google-generativeai==0.8.0
pillow==10.4.0
requests==2.32.3
numpy==1.26.4
scikit-learn==1.5.1
"""

with open("/content/hf_deployment/requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt written!")

✅ requirements.txt written!


In [11]:
# ============================================================
# STEP 2D: Upload model to HuggingFace Hub
# Username: abhianirudh — already set
# ============================================================

from huggingface_hub import HfApi, login
from google.colab import userdata
import os

# ---- Login ----
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Logged in via Colab Secrets!")
except:
    HF_TOKEN = input("Paste your HuggingFace WRITE token (hf_...): ").strip()
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Logged in!")

# ---- Config — already set for you ----
YOUR_HF_USERNAME = "abhianirudh"
MODEL_REPO_ID    = f"{YOUR_HF_USERNAME}/misinformation-guard-text"
SPACE_REPO_ID    = f"{YOUR_HF_USERNAME}/misinformation-guard"
TEXT_MODEL_PATH  = "/content/drive/MyDrive/MisinformationGuard/model_v2"

api = HfApi()

# ---- Create model repo ----
print(f"\n⏳ Creating model repo: {MODEL_REPO_ID}")
api.create_repo(
    repo_id  = MODEL_REPO_ID,
    token    = HF_TOKEN,
    exist_ok = True,
    private  = False,
)
print(f"✅ Repo ready!")

# ---- Check files ----
print(f"\n📁 Model files:")
for f in os.listdir(TEXT_MODEL_PATH):
    size = os.path.getsize(f"{TEXT_MODEL_PATH}/{f}") / 1e6
    print(f"   {f:45s} {size:6.1f} MB")

# ---- Upload ----
print(f"\n⏳ Uploading model (~1.1GB) — takes 3-8 min...")
api.upload_folder(
    folder_path = TEXT_MODEL_PATH,
    repo_id     = MODEL_REPO_ID,
    token       = HF_TOKEN,
)

print(f"""
✅ Model uploaded!

   View  : https://huggingface.co/{MODEL_REPO_ID}
   Space : https://huggingface.co/spaces/{SPACE_REPO_ID}

   Your permanent API URL (after deployment):
   https://abhianirudh-misinformation-guard.hf.space
""")

# Save for next steps
os.makedirs("/content/hf_deployment", exist_ok=True)
with open("/content/hf_deployment/model_id.txt", "w") as f:
    f.write(MODEL_REPO_ID)
with open("/content/hf_deployment/username.txt", "w") as f:
    f.write(YOUR_HF_USERNAME)

print("✅ Config saved — ready for Step 2E!")

✅ Logged in via Colab Secrets!

⏳ Creating model repo: abhianirudh/misinformation-guard-text
✅ Repo ready!

📁 Model files:
   config.json                                      0.0 MB
   model.safetensors                             1112.2 MB
   tokenizer_config.json                            0.0 MB
   tokenizer.json                                  17.1 MB
   training_metrics.json                            0.0 MB

⏳ Uploading model (~1.1GB) — takes 3-8 min...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...odel_v2/model.safetensors:   1%|          | 7.93MB / 1.11GB            

  ...d/model_v2/tokenizer.json:  47%|####6     | 7.96MB / 17.1MB            

No files have been modified since last commit. Skipping to prevent empty commit.



✅ Model uploaded!

   View  : https://huggingface.co/abhianirudh/misinformation-guard-text
   Space : https://huggingface.co/spaces/abhianirudh/misinformation-guard

   Your permanent API URL (after deployment):
   https://abhianirudh-misinformation-guard.hf.space

✅ Config saved — ready for Step 2E!


In [12]:
# ============================================================
# STEP 2E: Update app.py with your real model ID
# ============================================================

# Read existing app.py
with open("/content/hf_deployment/app.py", "r") as f:
    content = f.read()

# Replace placeholder with your real model ID
content = content.replace(
    "your-hf-username/misinformation-guard-text",
    "abhianirudh/misinformation-guard-text"
)

# Write back
with open("/content/hf_deployment/app.py", "w") as f:
    f.write(content)

print("✅ app.py updated!")
print("   Model ID: abhianirudh/misinformation-guard-text")

# Verify
with open("/content/hf_deployment/app.py", "r") as f:
    for line in f:
        if "MODEL_ID" in line or "abhianirudh" in line:
            print(f"   Verified: {line.strip()}")

✅ app.py updated!
   Model ID: abhianirudh/misinformation-guard-text
   Verified: MODEL_ID   = os.environ.get("TEXT_MODEL_ID", "abhianirudh/misinformation-guard-text")
   Verified: tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID)
   Verified: text_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)


In [13]:
# ============================================================
# STEP 3: Push Dockerfile + app.py + requirements.txt
# to your HuggingFace Space
# ============================================================

from huggingface_hub import HfApi
import os

api              = HfApi()
HF_TOKEN         = userdata.get("HF_TOKEN")
SPACE_REPO_ID    = "abhianirudh/misinformation-guard"

print(f"⏳ Pushing files to Space: {SPACE_REPO_ID}\n")

files_to_upload = ["app.py", "Dockerfile", "requirements.txt"]

for filename in files_to_upload:
    filepath = f"/content/hf_deployment/{filename}"

    if not os.path.exists(filepath):
        print(f"   ❌ {filename} not found — skipping")
        continue

    api.upload_file(
        path_or_fileobj = filepath,
        path_in_repo    = filename,
        repo_id         = SPACE_REPO_ID,
        repo_type       = "space",
        token           = HF_TOKEN,
    )
    print(f"   ✅ {filename} uploaded")

print(f"""
✅ ALL FILES PUSHED TO SPACE!

   Your Space is now building at:
   https://huggingface.co/spaces/abhianirudh/misinformation-guard

   Building takes 5-10 minutes.
   Watch the build log on that page.

   ⚠️  IMPORTANT: Add your Gemini key to the Space!
   Go to → Space Settings → Variables and Secrets
   Add secret: GEMINI_API_KEY = your key
""")

No files have been modified since last commit. Skipping to prevent empty commit.


⏳ Pushing files to Space: abhianirudh/misinformation-guard



No files have been modified since last commit. Skipping to prevent empty commit.


   ✅ app.py uploaded


No files have been modified since last commit. Skipping to prevent empty commit.


   ✅ Dockerfile uploaded
   ✅ requirements.txt uploaded

✅ ALL FILES PUSHED TO SPACE!

   Your Space is now building at:
   https://huggingface.co/spaces/abhianirudh/misinformation-guard

   Building takes 5-10 minutes.
   Watch the build log on that page.

   ⚠️  IMPORTANT: Add your Gemini key to the Space!
   Go to → Space Settings → Variables and Secrets
   Add secret: GEMINI_API_KEY = your key



In [14]:
# ============================================================
# FIX 1: Update CORS settings in app.py
# Allow requests from React localhost AND any deployed domain
# ============================================================

with open("/content/hf_deployment/app.py", "r") as f:
    content = f.read()

# Replace the existing CORS middleware with this permissive version
old_cors = '''app.add_middleware(
    CORSMiddleware,
    allow_origins  = ["*"],
    allow_methods  = ["*"],
    allow_headers  = ["*"],
)'''

new_cors = '''app.add_middleware(
    CORSMiddleware,
    allow_origins     = ["*"],
    allow_credentials = False,
    allow_methods     = ["GET", "POST", "PUT", "DELETE", "OPTIONS"],
    allow_headers     = ["*"],
    expose_headers    = ["*"],
    max_age           = 3600,
)'''

content = content.replace(old_cors, new_cors)

with open("/content/hf_deployment/app.py", "w") as f:
    f.write(content)

print("✅ CORS fixed!")

✅ CORS fixed!
